# LAB | Ensemble Methods

**Load the data**

In this challenge, we will be working with the same Spaceship Titanic data, like the previous Lab. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In this Lab, you should try different ensemble methods in order to see if can obtain a better model than before. In order to do a fair comparison, you should perform the same feature scaling, engineering applied in previous Lab.

In [1]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


Now perform the same as before:
- Feature Scaling
- Feature Selection


In [3]:
#your code here
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

In [26]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

**Perform Train Test Split**

In [17]:
#your code here

# Target
y = spaceship["Transported"].astype(int)

# Features - Feature Selection: Dropping columns we will not use
X = spaceship.drop(columns=["Transported", "PassengerId", "Name"])  # Keeping Cabin for now, as we will engineer Deck

# Feature engineering - Deck from Cabin
X["Deck"] = X["Cabin"].astype(str).str[0]
X = X.drop(columns=["Cabin"])  # Cabin is too granular, we are keeping Deck

# Identifying column types for preprocessing
num_cols = X.select_dtypes(include=["int64", "float64"]).columns
cat_cols = X.select_dtypes(include=["object", "bool"]).columns

# Preprocessing pipelines
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", num_pipeline, num_cols),
    ("cat", cat_pipeline, cat_cols)
])


**Model Selection** - now you will try to apply different ensemble methods in order to get a better model

In [13]:
# IMporting libraries
from sklearn.metrics import accuracy_score, classification_report
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier

- Bagging and Pasting

In [27]:
bagging_model = Pipeline([
    ("prep", preprocessor),
    ("model", BaggingClassifier(
        estimator=DecisionTreeClassifier(random_state=42),
        n_estimators=300,
        bootstrap=True,      # True = Bagging, False = Pasting
        random_state=42,
        n_jobs=-1
    ))
])

bagging_model.fit(X_train, y_train)
bag_pred = bagging_model.predict(X_test)

bag_acc = accuracy_score(y_test, bag_pred)
print("Bagging Accuracy:", bag_acc)
print(classification_report(y_test, bag_pred))


Bagging Accuracy: 0.79700977573318
              precision    recall  f1-score   support

           0       0.80      0.80      0.80       863
           1       0.80      0.80      0.80       876

    accuracy                           0.80      1739
   macro avg       0.80      0.80      0.80      1739
weighted avg       0.80      0.80      0.80      1739



- Random Forests

In [28]:
#your code here
rf_model = Pipeline([
    ("prep", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=500,
        random_state=42,
        n_jobs=-1
    ))
])

rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)

rf_acc = accuracy_score(y_test, rf_pred)
print("Random Forest Accuracy:", rf_acc)
print(classification_report(y_test, rf_pred))


Random Forest Accuracy: 0.7906843013225991
              precision    recall  f1-score   support

           0       0.78      0.81      0.79       863
           1       0.81      0.77      0.79       876

    accuracy                           0.79      1739
   macro avg       0.79      0.79      0.79      1739
weighted avg       0.79      0.79      0.79      1739



- Gradient Boosting

In [29]:
#your code here
gb_model = Pipeline([
    ("prep", preprocessor),
    ("model", GradientBoostingClassifier(random_state=42))
])

gb_model.fit(X_train, y_train)
gb_pred = gb_model.predict(X_test)

gb_acc = accuracy_score(y_test, gb_pred)
print("Gradient Boosting Accuracy:", gb_acc)
print(classification_report(y_test, gb_pred))


Gradient Boosting Accuracy: 0.8039102932719954
              precision    recall  f1-score   support

           0       0.82      0.77      0.80       863
           1       0.79      0.83      0.81       876

    accuracy                           0.80      1739
   macro avg       0.80      0.80      0.80      1739
weighted avg       0.80      0.80      0.80      1739



- Adaptive Boosting

In [30]:
#your code here
ada_model = Pipeline([
    ("prep", preprocessor),
    ("model", AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=1, random_state=42),
        n_estimators=300,
        learning_rate=0.5,
        random_state=42
    ))
])

ada_model.fit(X_train, y_train)
ada_pred = ada_model.predict(X_test)

ada_acc = accuracy_score(y_test, ada_pred)
print("AdaBoost Accuracy:", ada_acc)
print(classification_report(y_test, ada_pred))


c:\Users\jdhernandezd\AppData\Local\anaconda3\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(


AdaBoost Accuracy: 0.7929844738355377
              precision    recall  f1-score   support

           0       0.80      0.78      0.79       863
           1       0.79      0.81      0.80       876

    accuracy                           0.79      1739
   macro avg       0.79      0.79      0.79      1739
weighted avg       0.79      0.79      0.79      1739



Which model is the best and why?

In [37]:
results = {
    "Bagging": bag_acc,
    "Random Forest": rf_acc,
    "Gradient Boosting": gb_acc,
    "AdaBoost": ada_acc
}

results


{'Bagging': 0.79700977573318,
 'Random Forest': 0.7906843013225991,
 'Gradient Boosting': 0.8039102932719954,
 'AdaBoost': 0.7929844738355377}

In [38]:
best_model = max(results, key=results.get)
best_model, results[best_model]


('Gradient Boosting', 0.8039102932719954)

### Best Model

The best-performing model was **Gradient Boosting** with an accuracy of **0.80** on the test set.

This model performed best because it combines multiple models, which improves accuracy and helps reduce overfitting.

- **Random Forest / Bagging** improve stability by averaging many trees, which reduces variance.
- **Boosting (Gradient Boosting / AdaBoost)** improves performance by combining weak learners sequentially, which reduces bias.

Based on these results, **Gradient Boosting** provided the best outcome in terms of accuracy and overall performance across classes.
